In [6]:
# import os
# import sys
# import math
# import logging
# from pathlib import Path
import psignifit as pf
import psignifit.priors as priors

import plotly
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots

from tqdm import tqdm

import numpy as np
import scipy as sp
from scipy.stats import gamma, lognorm, norm

import sklearn
from sklearn.linear_model import LinearRegression as LinReg
import statsmodels.api as sm
from statsmodels.formula.api import ols

%load_ext autoreload
%autoreload 2

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


import pandas as pd
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Preprocessed --> Interim

"interim" data contains calculated values

In [2]:
# import os
# import sys
# import math
# import logging
# from pathlib import Path
import psignifit as pf
import psignifit.priors as priors

import plotly
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots

from tqdm import tqdm

import numpy as np
import scipy as sp
from scipy.stats import gamma, lognorm, norm

import sklearn
from sklearn.linear_model import LinearRegression as LinReg
import statsmodels.api as sm
from statsmodels.formula.api import ols

%load_ext autoreload
%autoreload 2

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


import pandas as pd
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

In [7]:
from psychoanalyze import data
from datetime import datetime

In [4]:
outlier_curve_ids = [108,101,870, 34]

In [5]:
df = pd.read_csv('../data/2-preprocessed/curves.csv')

In [22]:
df['Date'] = pd.to_datetime(df['Date'])
df.loc[df['Monkey']=='U','Days'] = (df['Date'] - datetime(2016,6,24)).dt.days
df.loc[df['Monkey']=='Y','Days'] = (df['Date'] - datetime(2016,3,11)).dt.days
df.loc[df['Monkey']=='Z','Days'] = (df['Date'] - datetime(2016,9,16)).dt.days


In [16]:
df = data.remove_outliers(df, 'curveID', outlier_curve_ids)
df = df.dropna()
df = data.add_relative_errors(df)
df = data.channelint2mask(df)
df = data.channelmask2num(df)

In [20]:
df.head()

,base,beta,beta_CI_5,beta_CI_95,curveID,gamma,gamma_CI_5,gamma_CI_95,lambda,lambda_CI_5,lambda_CI_95,location,location_CI_5,location_CI_95,width,width_CI_5,width_CI_95,Ref Amp,Ref PW,Active Channels,Return Channels,Monkey,Date,FAs,CRs,n_Amp,n_PW,Days,n_catch,FA_rate,varparam,err_upper_location,err_lower_location,err_upper_width,err_lower_width,err_upper_gamma,err_lower_gamma,err_upper_lambda,err_lower_lambda,err_upper_beta,err_lower_beta,Act Chan Mask,Ret Chan Mask,Channel(s)
0,200.0,1.628026e-07,0.370951,0.003926,1.0,1.631782e-01,0.260936,0.020754,1.152793e-02,0.050409,0.000936,680.935473,746.228485,534.736364,361.693192,1067.899224,263.593028,0.0,0.0,240.0,4.0,U,2016-10-05,24.0,245.0,8.0,1.0,103,269.0,0.089219,Amp,65.293011,146.199110,706.206031,98.100164,0.097758,0.142425,0.038881,0.010592,0.370951,-0.003926,11110000,00000100,6
1,200.0,1.052465e-09,0.275703,0.002229,4.0,2.073944e-02,0.117635,0.004530,1.259713e-14,0.038432,0.000386,735.304107,797.268666,675.015841,504.493879,725.639451,320.401527,0.0,0.0,240.0,2.0,U,2016-10-06,17.0,280.0,8.0,1.0,104,297.0,0.057239,Amp,61.964559,60.288266,221.145572,184.092352,0.096896,0.016210,0.038432,-0.000386,0.275703,-0.002229,11110000,00000010,7
2,200.0,5.483110e-14,0.388062,0.003421,5.0,4.180681e-02,0.182166,0.012805,3.171658e-02,0.106719,0.001276,787.940719,928.794973,709.102582,341.463810,1004.986934,257.070684,0.0,0.0,15.0,128.0,U,2016-10-07,8.0,86.0,8.0,1.0,105,94.0,0.085106,Amp,140.854254,78.838138,663.523124,84.393127,0.140359,0.029002,0.075003,0.030440,0.388062,-0.003421,00001111,10000000,1
3,200.0,1.708031e-04,0.346255,0.002920,6.0,6.845827e-09,0.110484,0.001031,7.600085e-09,0.040807,0.000297,751.497107,830.787458,687.537884,498.635910,755.088780,342.132500,0.0,0.0,15.0,16.0,U,2016-10-07,5.0,199.0,8.0,1.0,105,204.0,0.024510,Amp,79.290351,63.959223,256.452870,156.503410,0.110484,-0.001031,0.040807,-0.000297,0.346085,-0.002749,00001111,00010000,4
4,200.0,2.172097e-04,0.238075,0.002018,7.0,1.260665e-08,0.084078,0.000726,2.202251e-02,0.064036,0.008329,689.977447,751.977901,647.509029,428.493063,644.112290,298.110648,0.0,0.0,15.0,240.0,U,2016-10-10,25.0,320.0,8.0,1.0,108,345.0,0.072464,Amp,62.000454,42.468418,215.619228,130.382415,0.084078,-0.000726,0.042014,0.013694,0.237857,-0.001801,00001111,11110000,1+2+3+4


In [21]:
df.to_csv('../data/3-interim/curves.csv')